In [1]:
import json
import sys
from pathlib import Path
import re
import sympy

In [2]:
path = Path("test.json")
with path.open() as handle:
    raw = json.load(handle)

In [3]:
exprs = [str(expr).replace("^", "**") for expr in raw["equations"]]

In [4]:
print(exprs)

['-p1 * y1 + p2 * y2', 'p1 * y2 - p2 * y2 - p3 * y2', '-p1 * y1 + p2 * y2', 'p1 * y2 - p2 * y2 - p3 * y2', '-p1 * y1 + p2 * y2', 'p1 * y2 - p2 * y2 - p3 * y2']


In [5]:
ROW_LENGTH = 4
COLUMN_DISTANCE = "4cm"
ROW_DISTANCE = "3.5cm"

In [6]:
def collect_arrows(*arg):
    return None

In [7]:
def split_elements(equation):
    """'p1 * y1 - p3 * y2 + sin(y1) - p4' -> ['p1 * y1', '- p3 * y2', '+ sin(y1)', '- p4']"""
    elements = []
    sign = ""       # sign of the element being built: "", "+" or "-"
    current = ""    # text of the element being built, without its sign
    depth = 0       # how many brackets are open at this point
    for char in equation:
        if char == "(":
            depth += 1
        elif char == ")":
            depth -= 1
        elif char in "+-" and depth == 0:
            if not current.strip():
                # nothing collected yet: sign in front of the first element, e.g. '-p1 * y1'
                sign = char
                continue
            if current.rstrip()[-1] not in "*/^":
                # the previous element is complete: save it and start a new one
                elements.append(f"{sign} {current.strip()}".strip())
                sign, current = char, ""
                continue
            # otherwise the sign belongs to a factor, e.g. 'p1 * -y1', so it's kept below
        current += char
    elements.append(f"{sign} {current.strip()}".strip())
    return elements

def format_elements(elements):
    """[('+', 'p1 * y1'), ('-', 'p4')] -> ['p1 * y1', '- p4']"""
    row = [f"{sign} {body}" for sign, body in elements]
    row[0] = row[0].removeprefix("+ ")
    return row


def join_elements(elements):
    """[('+', 'p1 * y1'), ('-', 'p4')] -> 'p1 * y1 - p4'"""
    return " ".join(format_elements(elements))


def move_brackets_eq(equation):
    """'(p1 * y1 - p2 * y1) + p2 * y2 + p3 + p4' -> ['y1 * (p1 - p2)', '+ p2 * y2', '+ (p3 + p4)']"""
    elements = flatten(equation)
    elements = group_linear(elements)
    return format_elements(elements)

def create_matrix_for_eq(equations):
    matrix = []
    for equation in equations:
        equation = move_brackets_eq(equation) 
        # this function 1. removes unnecessary brackets 
        # '(p1 * y1 - p3 * y2) + sin(y1) - p4' -> 'p1 * y1 - p3 * y2 + sin(y1) - p4'
        # 2. groups linear elements with the same variable y_ together
        # 'p1 * y1 - p2 * y1 + p2 * y2 + p3 + p4' -> 'y1 * (p1 - p2) + p2 * y2 + (p3 + p4)'
        elements = split_elements(equation)
        matrix.append(elements)
    return matrix

In [8]:
def latex(text): #transforms equations to Latex compileable code
    text = text.replace(" ** ", "^").replace(" * ", " ")
    text = re.sub(r"\b([yp])(\d+)\b", r"\1_{\2}", text)
    return f"${text}$"


def render_tikz(system):
    equations = system["equations"]
    variables = [f"y{index}" for index in range(1, len(equations) + 1)]
    grid = {name: (index % ROW_LENGTH, index // ROW_LENGTH) for index, name in enumerate(variables)}
    arrows = collect_arrows(equations)

    lines = [r"\begin{tikzpicture}[auto, node distance=1cm]",]
    for index, name in enumerate(variables):
        box = f"$y_{{{index + 1}}}$"
        if index == 0:
            lines.append(f"    \\node [block] ({name}) {{{box}}};")
        elif index % ROW_LENGTH == 0:
            lines.append(f"    \\node [block, below of={variables[index - ROW_LENGTH]}, "
                         f"node distance={ROW_DISTANCE}] ({name}) {{{box}}};")
        else:
            lines.append(f"    \\node [block, right of={variables[index - 1]}, "
                         f"node distance={COLUMN_DISTANCE}] ({name}) {{{box}}};")
    lines.append("")

    if arrows != None:
        pairs = {(arrow["source"], arrow["target"]) for arrow in arrows if arrow["kind"] == "edge"}
        for arrow in arrows:
            if arrow["kind"] != "edge":
                continue
            source, target = arrow["source"], arrow["target"]
            label = latex(arrow["label"])
            column, row = grid[source]
            other_column, other_row = grid[target]
            adjacent = row == other_row and abs(column - other_column) == 1
            if (target, source) in pairs or not adjacent:
                lines.append(f"    \\draw [->] ({source}) to[bend left=20] "
                            f"node[sloped, anchor=south, font=\\scriptsize]{{{label}}} ({target});")
            else:
                lines.append(f"    \\draw [->] ({source}) --node[sloped, anchor=south, font=\\scriptsize]{{{label}}} ({target});")

    slots = {}
    if arrows != None:
        for arrow in arrows:
            if arrow["kind"] != "stub":
                continue
            box, direction = arrow["box"], arrow["direction"]
            slot = slots.get((box, direction), 0)
            slots[(box, direction)] = slot + 1
            offset = round(0.8 + 0.9 * slot, 2)
            label = latex(arrow["label"])
            if direction == "out":
                lines.append(f"    \\draw [<-] ($({offset}cm,1.5cm)+({box})$)"
                            f"node[above, font=\\scriptsize]{{{label}}} -- ({box});")
            else:
                lines.append(f"    \\draw [->] ($(-{offset}cm,-1.5cm)+({box})$)"
                            f"node[below, font=\\scriptsize]{{{label}}} -- ({box});")

    lines.append("\\end{tikzpicture}")
    return "\n".join(lines)

In [9]:
def render_eq_list(system):
    """Render the ODE system as a LaTeX align block, one row per equation."""
    rows = [
        rf"    \dot{{y}}_{{{i}}} &= {sympy.latex(sympy.sympify(eq))}"
        for i, eq in enumerate(system["equations"], start=1)
    ]
    return "\n".join([r"$\begin{aligned}", (r" \\" + "\n").join(rows), r"\end{aligned}$"])

In [10]:
def build_document(system):
    return "\n".join([
        r"\documentclass[border=10pt]{standalone}",
        r"\usepackage{amsmath}",
        r"\usepackage{tikz}",
        r"\usetikzlibrary{arrows.meta, positioning}",
        r"\tikzset{block/.style = {draw, rectangle, minimum size=1cm},>={Latex},}",
        r"\begin{document}",
        r"\begin{tabular}{c}",
        render_tikz(system),
        r"\\[1em]",
        render_eq_list(system),
        r"\end{tabular}",
        r"\end{document}",
        "",
    ])

In [14]:
print(raw["equations"])
print(raw["names"])

['-p1 * y1 + p2 * y2', 'p1 * y2 - p2 * y2 - p3 * y2', '-p1 * y1 + p2 * y2', 'p1 * y2 - p2 * y2 - p3 * y2', '-p1 * y1 + p2 * y2', 'p1 * y2 - p2 * y2 - p3 * y2']


KeyError: 'names'

In [12]:
latex = build_document(raw)
print(build_document(raw))

\documentclass[border=10pt]{standalone}
\usepackage{amsmath}
\usepackage{tikz}
\usetikzlibrary{arrows.meta, positioning}
\tikzset{block/.style = {draw, rectangle, minimum size=1cm},>={Latex},}
\begin{document}
\begin{tabular}{c}
\begin{tikzpicture}[auto, node distance=1cm]
    \node [block] (y1) {$y_{1}$};
    \node [block, right of=y1, node distance=4cm] (y2) {$y_{2}$};
    \node [block, right of=y2, node distance=4cm] (y3) {$y_{3}$};
    \node [block, right of=y3, node distance=4cm] (y4) {$y_{4}$};
    \node [block, below of=y1, node distance=3.5cm] (y5) {$y_{5}$};
    \node [block, right of=y5, node distance=4cm] (y6) {$y_{6}$};

\end{tikzpicture}
\\[1em]
$\begin{aligned}
    \dot{y}_{1} &= - p_{1} y_{1} + p_{2} y_{2} \\
    \dot{y}_{2} &= p_{1} y_{2} - p_{2} y_{2} - p_{3} y_{2} \\
    \dot{y}_{3} &= - p_{1} y_{1} + p_{2} y_{2} \\
    \dot{y}_{4} &= p_{1} y_{2} - p_{2} y_{2} - p_{3} y_{2} \\
    \dot{y}_{5} &= - p_{1} y_{1} + p_{2} y_{2} \\
    \dot{y}_{6} &= p_{1} y_{2} - p_{2} y_{